In [1]:
import pandas as pd
import json

In [2]:
def deal_query(query:str):
    query_final = query.replace("\t", " ")

    while "  " in query_final:
        query_final = query_final.replace("  ", " ")


    return query_final

def deal_nlqs(nlqs:list):
    for i in range(len(nlqs)):
        while "  " in nlqs[i]:
            nlqs[i] = nlqs[i].replace("  ", " ")

    nlqs_set = set([])

    nlqs_final = []

    for nlq in nlqs:
        if nlq.lower() not in nlqs_set:
            nlqs_set.add(nlq.lower())
            nlqs_final.append(nlq)

    return nlqs_final


In [3]:
data = pd.concat([pd.read_json("./spider/spider/train_spider.json", orient='records'), pd.read_json("./spider/spider/train_others.json", orient='records'), pd.read_json("./spider/spider/dev.json", orient='records')])

ban_db = ['sakila_1', 'store_1', 'baseball_1', 'wta_1', 'academic', 'geo', 'imdb', 'music_2', 'restaurants', 'scholar', 'yelp']

data_selected = data[['db_id', 'query', 'question']]
data_selected

,db_id,query,question
0,department_management,SELECT count(*) FROM head WHERE age > 56,How many heads of the departments are older th...
1,department_management,"SELECT name , born_state , age FROM head ORD...","List the name, born state and age of the heads..."
2,department_management,"SELECT creation , name , budget_in_billions ...","List the creation year, name and budget of eac..."
3,department_management,"SELECT max(budget_in_billions) , min(budget_i...",What are the maximum and minimum budget of the...
4,department_management,SELECT avg(num_employees) FROM department WHER...,What is the average number of employees of the...
...,...,...,...
1029,singer,SELECT Citizenship FROM singer WHERE Birth_Yea...,What are the citizenships that are shared by s...
1030,real_estate_properties,SELECT count(*) FROM Other_Available_Features,How many available features are there in total?
1031,real_estate_properties,SELECT T2.feature_type_name FROM Other_Availab...,What is the feature type name of feature AirCon?
1032,real_estate_properties,SELECT T2.property_type_description FROM Prope...,Show the property type descriptions of propert...


In [4]:


data_selected = data_selected[~data_selected['db_id'].isin(ban_db)]
# data_selected = data_selected[data_selected['db_id'] not in ban_db]



In [5]:

data_group = data_selected.groupby(['query'], as_index=False).agg(lambda x: list(x))

data_group['query'] = data_group['query'].apply(lambda q: deal_query(q))
data_group['db_id'] = data_group['db_id'].apply(lambda dbs: dbs[0])
data_group['question'] = data_group['question'].apply(lambda nlqs: deal_nlqs(nlqs))


# data_group = data_group.drop(data_group[data_group['db_id'] in ban_db].index)

data_group.to_json("./text2MongoDB_dataset/dataset.json", orient='records', indent=4)

len(data_group)

4308

In [6]:
total_length = data_group['question'].apply(len).sum()
total_length

7573